# 07 — Data Contract Check Pattern

Validate schema and required columns before processing.

Process:

INPUT DF
  |
  v
CONTRACT CHECK
  |
  +--> OK → continue
  |
  +--> FAIL → stop / log

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.getActiveSession()
if spark is not None:
    spark.stop()

spark = (
    SparkSession.builder
    .appName("data-contract-check")
    .master("spark://spark-master:7077")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

## Step 1 — Input

In [2]:
df = spark.createDataFrame(
    [("e1","u1",10.0)],
    ["event_id","user_id","amount"]
)

df.show()

+--------+-------+------+
|event_id|user_id|amount|
+--------+-------+------+
|      e1|     u1|  10.0|
+--------+-------+------+



## Step 2 — Contract definition

In [3]:
expected_columns = {"event_id","user_id","amount","event_time"}
required_columns = {"event_id","user_id"}

## Step 3 — Check missing columns

In [4]:
missing = expected_columns - set(df.columns)
missing

{'event_time'}

## Step 4 — Check required columns

In [5]:
missing_required = required_columns - set(df.columns)

if missing_required:
    raise Exception(f"Missing required columns: {missing_required}")

## Step 5 — Optional: drop extra columns

In [6]:
allowed_columns = expected_columns.intersection(set(df.columns))

df_clean = df.select(*allowed_columns)
df_clean.show()

+--------+-------+------+
|event_id|user_id|amount|
+--------+-------+------+
|      e1|     u1|  10.0|
+--------+-------+------+



## Step 6 — Control

In [7]:
print("columns:", df.columns)
print("valid:", len(missing_required) == 0)

columns: ['event_id', 'user_id', 'amount']
valid: True


In [8]:
spark.stop()